# Federated SegResNet32 training: FedAvg

Source domains act as simulated clients. Client updates are aggregated by training-cohort size, using the same patch sampling, augmentation, and loss settings as the centralized reference.

In [1]:
from pathlib import Path
import os
import sys
from functools import partial

_start = Path(os.environ.get("BRATS_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / "src" / "brats_pipeline").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Place notebooks/ and src/ under one project root, or set BRATS_PROJECT_ROOT.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
from brats_pipeline.common import ProjectPaths
paths_io = ProjectPaths(PROJECT_ROOT)
resolve_existing_path = paths_io.resolve
print("Project:", PROJECT_ROOT)

Project: /home/mandrakedrink/projects/research_ts_bs


In [ ]:
from brats_pipeline.common import seed_all, guard_output_directory
from brats_pipeline.data import (
    jp, load_nii as _load_nii, normalize_nonzero, remap_brats_labels,
    xyz_to_dhw, dhw_to_xyz, best_tumor_slice_dhw,
)
from brats_pipeline.inference import (
    create_segresnet, load_trusted_checkpoint, extract_model_state_dict,
    predict_labels_dhw, brats_region_dice_np, brats_region_voxels,
)
load_nii = partial(_load_nii, project_root=PROJECT_ROOT)
from brats_pipeline.training import (
    BratsManifestDatasetV2, WeightedDiceCELoss, dice_binary, brats_region_dice,
    validate_patch_level, get_round_lr, get_cpu_state_dict,
    fedavg_state_dicts, weighted_average_state_dicts,
    make_global_reference_state, fedprox_l2_penalty,
)

ALLOW_CHECKPOINT_OVERWRITE = False

## Imports

In [ ]:
import os
import json
import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nibabel as nib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.networks.nets import SegResNet
from monai.losses import DiceLoss

## Reproducibility and device

In [ ]:
seed_all(42, deterministic=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram gb:', torch.cuda.get_device_properties(0).total_memory / 1024 ** 3)

## Config

In [ ]:
CFG = {
    "manifest": str(PROJECT_ROOT / 'data/processed/manifest_4clients_seed42.csv'),

    # Input/training geometry.
    "patch_size": (128, 128, 128),
    "batch_size": 2,
    "grad_accum_steps": 2,
    "num_workers": 0,

    # Model.
    "num_classes": 4,
    "in_channels": 4,
    "init_filters": 32,

    # Optimization.
    "lr": 1e-4,
    "min_lr": 1e-6,
    "weight_decay": 1e-5,
    "grad_clip_norm": 12.0,

    # Federated training.
    "fed_rounds": 120,              # max rounds, not fixed final length
    "min_rounds": 40,
    "early_stop_patience": 20,
    "early_stop_min_delta": 1e-4,
    "local_epochs": 1,

    # Debug option. Use None for full client training.
    "max_client_batches": None,

    # Patch-level checkpoint validation.
    # Full-volume validation is done separately in notebook 04.
    "max_val_batches": None,

    # Mixed + ET-aware sampling probabilities.
    # Remapped labels: 3 == original BraTS label 4 / ET.
    "p_et": 0.20,
    "p_tumor": 0.50,
    "p_foreground": 0.20,
    "p_random": 0.10,

    # Lightweight 3D augmentations.
    "use_augmentation": True,
    "flip_prob": 0.50,
    "intensity_prob": 0.30,
    "scale_range": (0.90, 1.10),
    "shift_range": (-0.10, 0.10),
    "noise_prob": 0.20,
    "noise_std": 0.03,

    # Weighted CE inside Dice + CE loss.
    # Remapped labels:
    #   0 = background
    #   1 = original label 1 / tumor core
    #   2 = original label 2 / edema
    #   3 = original label 4 / enhancing tumor
    "use_ce_weight": True,
    "ce_weight": [0.10, 1.00, 1.00, 1.25],
    "dice_loss_weight": 1.0,
    "ce_loss_weight": 1.0,

    "save_dir": "models/federated_fedavg_segresnet32_final",
}

guard_output_directory(CFG["save_dir"], allow_overwrite=ALLOW_CHECKPOINT_OVERWRITE)

cfg_path = os.path.join(CFG["save_dir"], "fedavg_final_config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)

print("saved config:", cfg_path)
CFG

## Load manifest and define clients

In [ ]:
mf_df = pd.read_csv(CFG["manifest"])

print("manifest:", mf_df.shape)
display(mf_df.head())

display(mf_df.groupby(["client_domain", "split"]).size())

train_df = mf_df[mf_df["split"] == "train"].copy()
val_df = mf_df[mf_df["split"] == "val"].copy()

client_domains = sorted(train_df["client_domain"].unique().tolist())

print("train:", train_df.shape)
print("val:", val_df.shape)
print("client domains:", client_domains)

train_df.groupby("client_domain").size()

## Build client datasets

In [ ]:
client_train_dfs = {
    domain: train_df[train_df["client_domain"] == domain].copy()
    for domain in client_domains
}

for domain, df in client_train_dfs.items():
    print(domain, df.shape)

## Build client DataLoaders

In [ ]:
client_loaders = {}
for domain, df in client_train_dfs.items():
    ds = BratsManifestDatasetV2(
        df,
        patch_size=CFG['patch_size'],
        crop_mode='mixed',
        augment=CFG['use_augmentation'],
        config=CFG,
        project_root=PROJECT_ROOT
    )
    loader = DataLoader(
        ds,
        batch_size=CFG['batch_size'],
        shuffle=True,
        num_workers=CFG['num_workers'],
        pin_memory=torch.cuda.is_available()
    )
    client_loaders[domain] = loader
print('clients:', list(client_loaders.keys()))
for domain, loader in client_loaders.items():
    batch = next(iter(loader))
    print(
        domain,
        'image:',
        tuple(batch['image'].shape),
        'mask:',
        tuple(batch['mask'].shape),
        'crop_source sample:',
        list(batch['crop_source'])[:4],
        'augmented:',
        batch['augmented']
    )

## Validation DataLoader

In [ ]:
val_ds = BratsManifestDatasetV2(
    val_df,
    patch_size=CFG['patch_size'],
    crop_mode='tumor',
    config=CFG,
    project_root=PROJECT_ROOT
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available()
)
batch = next(iter(val_loader))
print('val image:', batch['image'].shape)
print('val mask:', batch['mask'].shape)
print('labels:', torch.unique(batch['mask']))

## Client weights for FedAvg

In [ ]:
client_sizes = {
    domain: len(df)
    for domain, df in client_train_dfs.items()
}

total_train = sum(client_sizes.values())

client_weights = {
    domain: client_sizes[domain] / total_train
    for domain in client_domains
}

print("client sizes:", client_sizes)
print("client weights:", client_weights)
print("sum weights:", sum(client_weights.values()))

## Model factory

In [ ]:
def create_model():
    return create_segresnet({
        "spatial_dims": 3, "in_channels": CFG["in_channels"],
        "out_channels": CFG["num_classes"], "init_filters": CFG["init_filters"],
        "blocks_down": (1, 2, 2, 4), "blocks_up": (1, 1, 1), "dropout_prob": 0.1,
    })

In [ ]:
global_model = create_model().to(device)

total_params = sum(p.numel() for p in global_model.parameters())
print("params:", f"{total_params:,}")

## Loss, AMP, and communication-round learning-rate schedule

In [ ]:
if CFG.get("use_ce_weight", True):
    ce_weight = torch.tensor(
        CFG["ce_weight"],
        dtype=torch.float32,
        device=device,
    )
else:
    ce_weight = None

loss_fn = WeightedDiceCELoss(
    ce_weight=ce_weight,
    dice_weight=CFG["dice_loss_weight"],
    ce_loss_weight=CFG["ce_loss_weight"],
    include_background=True,
)

use_amp = torch.cuda.is_available()


print("loss:", loss_fn.__class__.__name__)
print("ce_weight:", ce_weight)
print("AMP:", use_amp)
print("lr round 1:", get_round_lr(1, CFG["fed_rounds"], CFG["lr"], CFG["min_lr"]))
print("lr final:", get_round_lr(CFG["fed_rounds"], CFG["fed_rounds"], CFG["lr"], CFG["min_lr"]))

## Local client training

In [ ]:
def train_local_model(
    model,
    loader,
    loss_fn,
    device,
    lr=1e-4,
    weight_decay=1e-5,
    local_epochs=1,
    max_batches=None,
):
    """Optimize a client model and return loss and sampling statistics."""
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp,
    )

    grad_accum_steps = int(CFG.get("grad_accum_steps", 1))
    grad_clip_norm = CFG.get("grad_clip_norm", None)

    losses = []
    crop_sources = []

    optimizer.zero_grad(set_to_none=True)
    accum_counter = 0

    for local_epoch in range(local_epochs):
        for step, batch in enumerate(loader):
            if max_batches is not None and step >= max_batches:
                break

            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(images)
                loss = loss_fn(logits, masks.unsqueeze(1))
                loss_for_backward = loss / grad_accum_steps

            scaler.scale(loss_for_backward).backward()
            accum_counter += 1

            should_step = (accum_counter % grad_accum_steps) == 0

            if should_step:
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=grad_clip_norm,
                    )

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            losses.append(float(loss.item()))

            if "crop_source" in batch:
                crop_sources.extend(list(batch["crop_source"]))

    # Flush remaining accumulated gradients if number of batches was not divisible
    # by grad_accum_steps.
    if accum_counter > 0 and (accum_counter % grad_accum_steps) != 0:
        if grad_clip_norm is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=grad_clip_norm,
            )

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    mean_loss = float(np.mean(losses)) if len(losses) > 0 else float("nan")
    crop_source_counts = dict(Counter(crop_sources))

    return mean_loss, crop_source_counts

## One-client smoke test

In [ ]:
test_domain = client_domains[0]
test_lr = get_round_lr(
    round_idx=1,
    total_rounds=CFG["fed_rounds"],
    base_lr=CFG["lr"],
    min_lr=CFG["min_lr"],
)

local_model = create_model().to(device)
local_model.load_state_dict(global_model.state_dict())

test_loss, test_crop_counts = train_local_model(
    model=local_model,
    loader=client_loaders[test_domain],
    loss_fn=loss_fn,
    device=device,
    lr=test_lr,
    weight_decay=CFG["weight_decay"],
    local_epochs=1,
    max_batches=2,
)

del local_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("test domain:", test_domain)
print("test lr:", test_lr)
print("test local loss:", test_loss)
print("test crop counts:", test_crop_counts)

## FedAvg training loop

In [ ]:
history = []
best_val_loss = float('inf')
best_mean_dice = -float('inf')
best_round = None
rounds_without_improvement = 0
for round_idx in range(1, CFG['fed_rounds'] + 1):
    print(f"\n========== FedAvg round {round_idx}/{CFG['fed_rounds']} ==========")
    round_lr = get_round_lr(
        round_idx=round_idx,
        total_rounds=CFG['fed_rounds'],
        base_lr=CFG['lr'],
        min_lr=CFG['min_lr']
    )
    print(f'round_lr={round_lr:.2e}')
    global_state = get_cpu_state_dict(global_model)
    client_states = {}
    client_losses = {}
    client_crop_counts = {}
    for domain in client_domains:
        print(f'\nClient: {domain}')
        local_model = create_model().to(device)
        local_model.load_state_dict(global_state, strict=True)
        local_loss, crop_counts = train_local_model(
            model=local_model,
            loader=client_loaders[domain],
            loss_fn=loss_fn,
            device=device,
            lr=round_lr,
            weight_decay=CFG['weight_decay'],
            local_epochs=CFG['local_epochs'],
            max_batches=CFG['max_client_batches']
        )
        client_losses[domain] = local_loss
        client_crop_counts[domain] = crop_counts
        client_states[domain] = get_cpu_state_dict(local_model)
        del local_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'{domain} local_loss={local_loss:.4f} | crops={crop_counts}')
    avg_state = fedavg_state_dicts(client_states=client_states, client_weights=client_weights)
    global_model.load_state_dict(avg_state, strict=True)
    global_model.to(device)
    val_loss, val_dice = validate_patch_level(
        model=global_model,
        loader=val_loader,
        loss_fn=loss_fn,
        device=device,
        max_batches=CFG['max_val_batches'],
        use_amp=use_amp
    )
    mean_dice = float(np.mean([val_dice['WT'], val_dice['TC'], val_dice['ET']]))
    row = {
        'round': round_idx,
        'lr': round_lr,
        'val_loss': val_loss,
        'val_dice_WT': val_dice['WT'],
        'val_dice_TC': val_dice['TC'],
        'val_dice_ET': val_dice['ET'],
        'val_mean_dice': mean_dice
    }
    for domain in client_domains:
        row[f'local_loss_{domain}'] = client_losses[domain]
        row[f'crop_source_counts_{domain}'] = json.dumps(client_crop_counts[domain])
    history.append(row)
    improved_mean_dice = mean_dice > best_mean_dice + CFG['early_stop_min_delta']
    if improved_mean_dice:
        best_mean_dice = mean_dice
        best_round = round_idx
        rounds_without_improvement = 0
    else:
        rounds_without_improvement += 1
    print(f"\nRound {round_idx:03d} | val_loss={val_loss:.4f} | mean_dice={mean_dice:.4f} | WT={val_dice['WT']:.4f} | TC={val_dice['TC']:.4f} | ET={val_dice['ET']:.4f} | no_improve={rounds_without_improvement}")
    ckpt_last_path = os.path.join(CFG['save_dir'], 'fedavg_last.pt')
    torch.save(
        {
            'model': global_model.state_dict(),
            'round': round_idx,
            'cfg': CFG,
            'history': history,
            'client_weights': client_weights,
            'client_sizes': client_sizes,
            'best_round': best_round,
            'best_mean_dice': best_mean_dice
        },
        ckpt_last_path
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_best_loss_path = os.path.join(CFG['save_dir'], 'fedavg_best_loss.pt')
        torch.save(
            {
                'model': global_model.state_dict(),
                'round': round_idx,
                'cfg': CFG,
                'history': history,
                'client_weights': client_weights,
                'client_sizes': client_sizes,
                'best_val_loss': best_val_loss,
                'best_round': best_round,
                'best_mean_dice': best_mean_dice
            },
            ckpt_best_loss_path
        )
        print('saved best loss:', ckpt_best_loss_path)
    if improved_mean_dice:
        ckpt_best_dice_path = os.path.join(CFG['save_dir'], 'fedavg_best_mean_dice.pt')
        torch.save(
            {
                'model': global_model.state_dict(),
                'round': round_idx,
                'cfg': CFG,
                'history': history,
                'client_weights': client_weights,
                'client_sizes': client_sizes,
                'best_mean_dice': best_mean_dice,
                'best_round': best_round
            },
            ckpt_best_dice_path
        )
        print('saved best mean dice:', ckpt_best_dice_path)
    if round_idx >= CFG['min_rounds'] and rounds_without_improvement >= CFG['early_stop_patience']:
        print(f'Early stopping at round {round_idx}. Best round: {best_round}, best_mean_dice={best_mean_dice:.4f}')
        break

## Save FL history

In [ ]:
hist = pd.DataFrame(history)

hist_path = os.path.join(
    CFG["save_dir"],
    "fedavg_history.csv",
)

hist.to_csv(hist_path, index=False)

print("saved:", hist_path)

hist

## Validation curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist["round"], hist["val_loss"], marker="o")
plt.xlabel("Federated round")
plt.ylabel("Patch-level validation loss")
plt.title("FedAvg patch-level validation loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist["round"], hist["val_dice_WT"], marker="o", label="WT")
plt.plot(hist["round"], hist["val_dice_TC"], marker="o", label="TC")
plt.plot(hist["round"], hist["val_dice_ET"], marker="o", label="ET")
plt.plot(hist["round"], hist["val_mean_dice"], marker="o", linestyle="--", label="Mean Dice")
plt.xlabel("Federated round")
plt.ylabel("Patch-level Dice")
plt.title("FedAvg patch-level validation Dice")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Client local losses

In [ ]:
plt.figure(figsize=(9, 5))

for domain in client_domains:
    col = f"local_loss_{domain}"
    plt.plot(hist["round"], hist[col], marker="o", label=domain)

plt.xlabel("Federated round")
plt.ylabel("Local training loss")
plt.title("Client local losses across FedAvg rounds")
plt.legend()
plt.grid(alpha=0.3)
plt.show()